In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')   

In [3]:
df =pd.read_csv(r"D:\DA\Archive\GitHUB\Forage-QA\Creditriskdata.csv")
df.head(10)

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0
5,4661159,0,5376.886873,7189.121298,85529.84591,2,697,0
6,8291909,1,3634.057471,7085.980095,68691.57707,6,722,0
7,4616950,4,3302.172238,13067.570210,50352.16821,3,545,1
8,3395789,0,2938.325123,1918.404472,53497.37754,4,676,0
9,4045948,0,5396.366774,5298.824524,92349.55399,2,447,0


In [4]:
fico = df['fico_score']
default = df['default']
df = df.sort_values('fico_score').reset_index(drop=True)

In [5]:
n = len(df)
cum_total = np.arange(1, n+1)
cum_default = df['default'].cumsum()
def log_likelihood(start, end):
    ni = end - start + 1
    ki = cum_default[end] - (cum_default[start-1] if start > 0 else 0)

    if ki == 0 or ki == ni:
        return 0 

    pi = ki / ni

    return ki * np.log(pi) + (ni - ki) * np.log(1 - pi)

In [8]:
K = 5  

dp = np.full((n, K+1), -np.inf)
split = np.zeros((n, K+1), dtype=int)

for i in range(n):
    dp[i][1] = log_likelihood(0, i)

MIN_BUCKET_SIZE = 50  # adjust based on dataset

for k in range(2, K+1):
    for i in range(n):
        for j in range(max(0, i - 500), i):  # LIMIT RANGE
            if (i - j) < MIN_BUCKET_SIZE:
                continue

            val = dp[j][k-1] + log_likelihood(j+1, i)

            if val > dp[i][k]:
                dp[i][k] = val
                split[i][k] = j

In [9]:
buckets = []
i = n - 1
k = K

while k > 0:
    j = split[i][k]
    buckets.append((j+1, i))
    i = j
    k -= 1

buckets.reverse()

In [10]:
rating_map = []

for idx, (start, end) in enumerate(buckets):
    subset = df.iloc[start:end+1]
    fico_min = subset['fico_score'].min()
    fico_max = subset['fico_score'].max()

    pd_value = subset['default'].mean()

    rating_map.append({
        "Rating": K - idx, 
        "FICO Range": (fico_min, fico_max),
        "PD": round(pd_value, 4)
    })

rating_df = pd.DataFrame(rating_map)
print(rating_df)

   Rating  FICO Range      PD
0       5  (409, 689)  0.2178
1       4  (689, 700)  0.0771
2       3  (700, 714)  0.0646
3       2  (714, 735)  0.0462
4       1  (735, 850)  0.0260
